# **PROJECT PHASE II**
**Rachel MacMillan // rem388** <br>
**Valentina Fernández // jvf48**

## **Research Question:** <br>
**How do patterns of leisure activities, including television viewing, social media use, reading, and exercise, differ across age groups, and how are these differences associated with cognitive and mental health assessment outcomes?**

Research on prolonged passive media consumption (such as binge-watching or "doomscrolling") shows that these activities negatively affect cognitive function and mental-wellbeing. Alternatively, active leisure activities such as reading and exercise are frequently associated with positive cognitive and psychological outcomes. 

Determining how leisure activities affect cognitive function and mental-wellbeing typically requires controlled experimental studies, which are difficult to conduct and beyond the scope of this project. Instead, we will investigate how leisure activity patterns vary across age groups and whether those patterns are associated with differences in cognitive and mental health assessments. 

By combining datasets that measure leisure activities, demographic characteristics, and cognitive or mental health outcomes, we hope to identify meaningful relationships between lifestyle behaviors and well-being. In particular, we are interested in whether age groups characterized by greater participation in passive screen-based activities differ from groups that spend more time reading or exercising with respect to measures such as distractibility, forgetfulness, and overall mental health. While our analysis will not establish causation, it may reveal patterns that help motivate future research on how different forms of leisure activity relate to cognitive and psychological well-being.

## **Datasets:**

**[2025 American Time Use Survey: Activity Summary File](https://www.bls.gov/tus/data/datafiles-2025.htm):** <br>
The U.S. Census Bureau annually conducts the American Time Use Survey, which measures how American's divide their time among daily activities. These activities are organized into major categories such as personal care, eating and drinking, household activities, work, education, leisure and sports, and communication, with each category further divided into more detailed subcategories. 

ATUS participants are randomly selected from a subset of households that have completed their eighth month of interviews for the U.S. Census Population Survey. Participants are then interviewed once to collect time-use data of their previous day. Demographic information such as age, sex, martial status, occupation, income, and education are also recorded for each participant. 

ATUS provides several microdatasets regarding information on the respondant, activity, and activity summary. We will use ATUS's [Activity Summary File](https://drive.google.com/drive/folders/1Etczii3Ya7hL1ITElyc8QnaYF7_iJU_3?usp=sharing) (our raw data), which includes demographical and time-use data for each of the 6,145 participants. 


**[Cognitive Assessment Questionnaire Dataset](https://data.mendeley.com/datasets/m2b8563ks5/1):** <br>
The Cognitive Assessment Questionnaire was designed to collect information about the frequency of everyday cognitive failures related to memory, perception, and attention. A total of 1,399 participants completed a 25-question online survey in which they reported how often they experienced specific cognitive lapses during the previous six months. Responses were recorded on a 5-point scale ranging from 0 (Never) to 4 (Very Often).

The questionnaire items are grouped into three dimensions of cognitive failure: Distractibility, Forgetfulness, and False Triggering. The dataset includes both the original survey responses and a processed version that classifies participants' levels of cognitive failure within each dimension as Low, Medium, or High.




## **Data Cleaning & Descriptions:**

In [2]:
!pip install pandas numpy duckdb

In [3]:
import pandas as pd
import numpy as np
import duckdb

### **Cleaning American Time Use Dataset**

In [4]:
atus_summ = pd.read_csv("data/atusSummary/atussum_2025.dat")
atus_summ.head()

,TUCASEID,TUFINLWGT,TRYHHCHILD,TEAGE,TESEX,PEEDUCA,PTDTRACE,PEHSPNON,GTMETSTA,TELFS,...,t181401,t181501,t181599,t181601,t181801,t189999,t500101,t500103,t500105,t500106
0,20250101250011,3.193152e+07,-1,55,1,39,1,2,2,1,...,0,0,0,0,0,0,0,0,0,0
1,20250101250092,3.583641e+07,-1,58,2,41,1,2,1,5,...,0,0,0,0,0,0,0,0,0,0
2,20250101250283,2.607856e+07,-1,24,1,39,1,2,1,1,...,0,0,0,0,0,0,0,0,0,15
3,20250101250305,3.535833e+07,-1,25,2,44,7,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,20250101250594,5.520668e+06,-1,80,2,39,1,2,1,5,...,0,0,0,0,0,0,0,0,0,0


We used ATUS's [User Guide](https://www.bls.gov/tus/atususersguide.pdf) to determine which leisure activity each lexicode variable represented. 

**relevant atus_summ variables:** 
- `TUCASEID`: unique identifier for each participant
- `TEAGE`: age of participant
- `t010101`: minutes spent sleeping per day
- `t120303`: minutes spent watching TV per day
- `t120307`: minutes spent playing games (such as board games, videogames, or mobile games) per day
- `t120308`: minutes spent using computer for leisure (such as websurfing or using social media; **excludes gaming**) per day
- `t120309`: minutes spent doing arts and crafts as a hobby per day
- `t120312`: minutes spent reading for personal interest per day
- `t120313`: minutes spent writing for personal interest per day
- `t130101`: minutes spent exercising (working out or playing sports) per day

We will now create a new dataframe `activities` which includes the relevant variables and we will rename them appropriately. 

In [5]:
#activites shows the age of each participant and the minutes they spent on each activity per day
activities = pd.DataFrame({
    "id" : atus_summ["TUCASEID"],
    "age" : atus_summ["TEAGE"],
    "sleeping" : atus_summ["t010101"],
    "watchingTV" : atus_summ["t120303"],
    "gaming" : atus_summ["t120307"],
    "onComputer" : atus_summ["t120308"],
    "artsCrafts" : atus_summ["t120309"],
    "reading" : atus_summ["t120312"],
    "writing" : atus_summ["t120313"],
    "exercising" : atus_summ["t130101"]
})
print("activites: the average minutes each participant spent on each activity per day")
activities.head(10)


activites: the average minutes each participant spent on each activity per day


,id,age,sleeping,watchingTV,gaming,onComputer,artsCrafts,reading,writing,exercising
0,20250101250011,55,540,30,0,0,0,0,0,0
1,20250101250092,58,725,270,0,0,0,0,0,0
2,20250101250283,24,390,83,0,0,0,0,0,0
3,20250101250305,25,580,0,0,0,0,135,0,0
4,20250101250594,80,840,290,0,0,0,0,0,0
5,20250101250600,67,660,0,0,0,0,0,0,0
6,20250101250630,62,615,180,0,0,0,0,0,0
7,20250101250640,72,690,265,0,0,0,30,0,0
8,20250101250657,78,565,60,0,0,0,0,0,0
9,20250101250816,46,850,0,0,0,0,0,0,0


As a sanity check, we will compare our statistics with [ATUS's Lastest Numbers](https://www.bls.gov/tus/latest-numbers.htm), specifically ATUS reports that American's spent on average 9.03 hours sleeping in 2025. 

In [6]:
avg_hours_sleeping = (activities["sleeping"].mean())/60
print("Average Hours Sleeping in 2025: " + str(round(avg_hours_sleeping, 2)))


Average Hours Sleeping in 2025: 9.03


Now we will create `avg_activities` which will compute the average time spent on each activity for the population as a whole

In [7]:
avg_activities = duckdb.sql(""" SELECT ROUND(AVG(sleeping), 2) AS avg_sleeping, 
                                       ROUND(AVG(watchingTV), 2) AS avg_tv, 
                                       ROUND(AVG(gaming), 2) AS avg_gaming, 
                                       ROUND(AVG(onComputer), 2) AS avg_onComputer, 
                                       ROUND(AVG(artsCrafts), 2) AS avg_artsCrafts, 
                                       ROUND(AVG(reading), 2) AS avg_reading, 
                                       ROUND(AVG(writing), 2) AS avg_writing, 
                                       ROUND(AVG(exercising), 2) AS avg_exercising 
                                       FROM activities """).df()
print("Average minutes spent on each activity per day (full population):")
avg_activities.head()

Average minutes spent on each activity per day (full population):


,avg_sleeping,avg_tv,avg_gaming,avg_onComputer,avg_artsCrafts,avg_reading,avg_writing,avg_exercising
0,541.73,185.11,20.1,11.37,2.7,20.98,0.35,0.12


Next, we will create `grouped_activities` which is `activities` grouped by the following age categories:
- 15 - 24
- 25 - 34
- 35 - 44
- 45 - 54
- 55 - 64
- 65+

In [8]:
#avg_grouped_activies shows the average minutes spent doing each activity per day for each age group
avg_grouped_activities = duckdb.sql(""" SELECT age_group,
                                               COUNT(*) AS num_participants,
                                               ROUND(AVG(sleeping), 2) AS avg_sleeping,
                                               ROUND(AVG(watchingTV), 2) AS avg_tv,
                                               ROUND(AVG(gaming), 2) AS avg_gaming,
                                               ROUND(AVG(onComputer), 2) AS avg_computer,
                                               ROUND(AVG(artsCrafts), 2) AS avg_artsCrafts,
                                               ROUND(AVG(reading), 2) AS avg_reading,
                                               ROUND(AVG(writing), 2) AS avg_writing,
                                               ROUND(AVG(exercising), 2) AS avg_exercising
                                    FROM (
                                        SELECT
                                            CASE
                                                WHEN age BETWEEN 15 AND 24 THEN '15-24'
                                                WHEN age BETWEEN 25 AND 44 THEN '25-44'
                                                WHEN age BETWEEN 45 AND 64 THEN '45-64'
                                                ELSE '65+'
                                            END AS age_group,
                                            sleeping,
                                            watchingTV,
                                            gaming,
                                            onComputer,
                                            artsCrafts,
                                            reading,
                                            writing,
                                            exercising
                                        FROM activities
                                    )
                                    GROUP BY age_group
                                    ORDER BY age_group """).df()

print("avg_grouped_activities: the average minutes spent on each activity per day for each age group")
avg_grouped_activities.head()

avg_grouped_activities: the average minutes spent on each activity per day for each age group


,age_group,num_participants,avg_sleeping,avg_tv,avg_gaming,avg_computer,avg_artsCrafts,avg_reading,avg_writing,avg_exercising
0,15-24,440,610.01,115.74,54.35,19.20,3.72,10.71,0.30,0.14
1,25-44,1749,541.45,119.64,25.13,10.41,1.45,12.75,0.17,0.12
2,45-64,1827,531.52,175.18,10.71,10.08,1.95,15.44,0.36,0.12
3,65+,2130,536.60,261.70,16.94,11.64,4.16,34.61,0.50,0.12


We are also curious about the age distribution in this sample. We will create `age_dist_activities` to illustrate the distribution. 

In [9]:
age_dist_activities = duckdb.sql(""" SELECT age_group, 
                                            num_participants,
                                            ROUND((num_participants * 100.0 / SUM(num_participants) OVER()), 2) AS percentage_of_population
                                     FROM avg_grouped_activities
                                     GROUP BY age_group, num_participants
                                     ORDER BY age_group """).df()
age_dist_activities.head()

,age_group,num_participants,percentage_of_population
0,15-24,440,7.16
1,25-44,1749,28.46
2,45-64,1827,29.73
3,65+,2130,34.66


### **Data Description for `avg_activities` and `avg_grouped_activities`:**

`avg_activities`:

`avg_grouped_activities`:

### **Cleaning Cognitive Assessment Questionnaire Dataset**

First we will create `questionnaire`, which will consist of participant information and responses to the cognitive questionnaire.

In [10]:
questionnaire = pd.read_csv("data/cognitiveAssessment/Cognitive_Assessment_Anonymized_Responses.csv")
print("questionnaire: the responses from the cognitive assessment questionnaire")
questionnaire.head()

questionnaire: the responses from the cognitive assessment questionnaire


,Participant_ID,Occupation,Age,Gender,Education,Marks_10th,Marks_12th,Q1,Q2,Q3,...,Q16,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25
0,P0001,Student,27,Female,M. Tech,73.2,53.34,2,4,4,...,2,4,0,1.0,4,3,4,3,0,1
1,P0002,Business,31,Male,B Tech (agri),0.6,0.55,1,0,2,...,1,1,0,0.0,1,0,1,0,0,1
2,P0003,Business,36,Male,MBA,60,62,3,0,0,...,1,2,0,1.0,2,1,2,0,1,1
3,P0004,Student,23,Male,Bsc,85,75,0,0,0,...,0,0,0,0.0,0,0,0,0,0,1
4,P0005,other add this feature,29,Female,M pharm,75 %,0.61,1,1,0,...,0,0,1,1.0,1,1,1,0,1,1


Then we will create `processed_data` which includes evalutations for the forgetfulness, distractibility, and `False Triggering categories.

In [11]:
processed_data = pd.read_csv("data/cognitiveAssessment/Cognitive_Assessment_Processed_Data.csv")
print("processed_data: the processed data from the cognitive assessment")
processed_data.head()

processed_data: the processed data from the cognitive assessment


,Participant_ID,Q1,Q2,Q3,Q4,Q5,Q6,Q7,Q8,Q9,...,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Forgetfulness_Category,Distractibility_Category,False_Triggering_Category
0,P0001,2,4,4,4,3,4,1,0,2,...,1.0,4,3,4,3,0,1,Mild,High,No
1,P0002,1,0,2,1,2,1,0,1,1,...,0.0,1,0,1,0,0,1,Mild,Low,Yes
2,P0003,3,0,0,0,1,0,2,2,2,...,1.0,2,1,2,0,1,1,Mild,High,Yes
3,P0004,0,0,0,0,0,0,0,1,0,...,0.0,0,0,0,0,0,1,Mild,High,No
4,P0005,1,1,0,0,0,1,0,0,0,...,1.0,1,1,1,0,1,1,Mild,Low,Yes


We are interested in each participant's age and their measures of forgetfulness, distractibility, and false triggering. So we will create `cognitive_data`, which will combine the relevant variables in the `questionnaire` and `processed_data` datasets.

In [12]:
cognitive_data = duckdb.sql(""" SELECT questionnaire.Participant_ID AS id,
                                       questionnaire.Age AS age,
                                       processed_data.Forgetfulness_Category AS forgetfulness,
                                       processed_data.Distractibility_Category AS distractibility,
                                       processed_data.False_Triggering_Category AS false_triggering
                                FROM questionnaire 
                                INNER JOIN processed_data 
                                ON questionnaire.Participant_ID = processed_data.Participant_ID """).df()
cognitive_data.head()


,id,age,forgetfulness,distractibility,false_triggering
0,P0001,27,Mild,High,No
1,P0002,31,Mild,Low,Yes
2,P0003,36,Mild,High,Yes
3,P0004,23,Mild,High,No
4,P0005,29,Mild,Low,Yes


We then will ensure that our data is of the correct type.

In [13]:
print("Original data types of cognitive_data:")
print(cognitive_data.dtypes)

#convert age to int by extracting the numeric part of the string and converting it to int
cognitive_data["age"] = (
    cognitive_data["age"].str.extract(r"(\d+)")
    .astype(int)
)

print()
print("Data types after converting age to int:")
print(cognitive_data.dtypes)

Original data types of cognitive_data:
id                  str
age                 str
forgetfulness       str
distractibility     str
false_triggering    str
dtype: object

Data types after converting age to int:
id                    str
age                 int64
forgetfulness         str
distractibility       str
false_triggering      str
dtype: object


Because we cannot easily get the average of the categorical data, we group `cognitive_data` by the same age group as we did with `avg_grouped_activities` and aggregate by the counts of each measure for each category.

In [14]:
grouped_cognitive_data = duckdb.sql(""" SELECT age_group, 
                                               COUNT(*) AS num_participants,
                                               ROUND(100.0 * COUNT(CASE WHEN forgetfulness = 'Mild' THEN 1 END) / COUNT(*), 2) AS low_forgetfulness_percentage,
                                               ROUND(100.0 * COUNT(CASE WHEN forgetfulness = 'Moderate' THEN 1 END) / COUNT(*), 2) AS medium_forgetfulness_percentage,
                                               ROUND(100.0 * COUNT(CASE WHEN forgetfulness = 'High' THEN 1 END) / COUNT(*), 2) AS high_forgetfulness_percentage,
                                              
                                               ROUND(100.0 * COUNT(CASE WHEN distractibility = 'Low' THEN 1 END) / COUNT(*), 2) AS low_distractibility_percentage,
                                               ROUND(100.0 * COUNT(CASE WHEN distractibility = 'Medium' THEN 1 END) / COUNT(*), 2) AS medium_distractibility_percentage,
                                               ROUND(100.0 * COUNT(CASE WHEN distractibility = 'High' THEN 1 END) / COUNT(*), 2) AS high_distractibility_percentage,
                                              
                                               ROUND(100.0 * COUNT(CASE WHEN false_triggering = 'Yes' THEN 1 END) / COUNT(*), 2) AS yes_false_triggering_percentage,
                                               ROUND(100.0 * COUNT(CASE WHEN false_triggering = 'None' THEN 1 END) / COUNT(*), 2) AS no_false_triggering_percentage
                                       FROM (
                                            SELECT
                                                CASE
                                                    WHEN age BETWEEN 15 AND 24 THEN '15-24'
                                                    WHEN age BETWEEN 25 AND 44 THEN '25-44'
                                                    WHEN age BETWEEN 45 AND 64 THEN '45-64'
                                                    ELSE '65+'
                                                END AS age_group,
                                            forgetfulness,
                                            distractibility,
                                            false_triggering
                                        FROM cognitive_data
                                    )
                                    GROUP BY age_group
                                    ORDER BY age_group """).df()
grouped_cognitive_data.head()

,age_group,num_participants,low_forgetfulness_percentage,medium_forgetfulness_percentage,high_forgetfulness_percentage,low_distractibility_percentage,medium_distractibility_percentage,high_distractibility_percentage,yes_false_triggering_percentage,no_false_triggering_percentage
0,15-24,388,88.92,7.73,3.35,32.99,31.44,35.57,52.58,0.0
1,25-44,960,90.63,6.67,2.71,35.21,33.23,31.56,49.27,0.0
2,45-64,50,86.00,12.00,2.00,34.00,30.00,36.00,50.00,0.0
3,65+,1,100.00,0.00,0.00,0.00,100.00,0.00,100.00,0.0


We will create `age_dist_cognitive_data` to illustrate the age distribution of the sample. 

In [15]:
age_dist_cognitive_data = duckdb.sql(""" SELECT age_group, 
                                                num_participants,
                                                ROUND((num_participants * 100.0 / SUM(num_participants) OVER()), 2) AS percentage_of_population
                                         FROM grouped_cognitive_data
                                         GROUP BY age_group, num_participants
                                         ORDER BY age_group """).df()
age_dist_cognitive_data.head()

,age_group,num_participants,percentage_of_population
0,15-24,388,27.73
1,25-44,960,68.62
2,45-64,50,3.57
3,65+,1,0.07


### **Data Description for `grouped_cognitive_data`:**

`grouped_cognitive_data` ...

# **Limitations:**

# **Exploratory Data Analysis:**

# **Questions for Reviewers:**